# 04_04 Connecting attention and feed-forward in a block

*📖 Book: section 4.5*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
>
> **This notebook:** The parts so far (attention, feed-forward, layer norms, shortcuts) are loose. How do they fit into one block that can be stacked?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How the four parts fit into one block: norm, attention, shortcut, norm, feed-forward, shortcut
- How to write the block as a class, built here from small pieces
- That the shape going in equals the shape coming out
- How many weights one block has, part by part

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **output projection** | one extra layer after the heads are joined, so the heads can mix their results | 32 × 32 weights, plus 32 biases |
| **buffer** | a table a model keeps with it, saved and moved with it, but which isn't learned | the causal mask |

## 3. The idea

### ❓ In which order do the parts go?

A block takes a row of numbers for each letter and gives back rows of the same shape. Inside there are two steps, and both follow the same recipe: **norm first, then the part, then add the input back** (a shortcut, [04_03](04_03_shortcuts.ipynb)).

Putting the norm before each part (not after) is what GPT-2 and the book do. It's called pre-norm. Dropout (switching off some numbers at random while training, [03_03](../03_attention/03_03_hiding_the_future.ipynb)) is optional. With `drop_rate` 0 it does nothing.

**Look at:** the grey road goes up through the parts. The green road is the shortcut that goes round them and is added back at each `+`.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    plus,
    studio_figure,
    tag,
)

fig = studio_figure("One block", "Two steps, each with a shortcut", (8, 9.5))
ax = diagram_axes(fig)
X, SIDE, PITCH = 4.0, 7.0, 0.85  # the main road, the shortcut road, box spacing


def step(base, parts, notes, carried):
    # one step: the road splits at `base`; the parts go up the main road;
    # the shortcut goes round them and joins again at the + on top
    top = base + 0.8 + len(parts) * PITCH
    low = base
    for i, (name, note) in enumerate(zip(parts, notes, strict=True)):
        y = base + 0.8 + i * PITCH
        box(ax, X, y, 2.4, 0.55, name, fontsize=10)
        tag(ax, X - 1.4, y, note, MUTED, 8, ha="right")
        arrow(ax, (X, low), (X, y - 0.3), head=9)
        low = y + 0.3
    arrow(ax, (X, low), (X, top - 0.27), head=9)
    plus(ax, X, top, 0.25)
    ax.plot([X, SIDE, SIDE], [base, base, top], color=GOOD, linewidth=2.2)
    arrow(ax, (SIDE, top), (X + 0.27, top), color=GOOD, width=2.2)
    ax.scatter([X], [base], s=40, color=GOOD, zorder=4)
    note = f"shortcut: {carried} goes round\nand is added back"
    tag(ax, SIDE + 0.2, (base + top) / 2, note, GOOD, 9, ha="left")
    return top


tag(ax, X, 0.3, "x   (1, 5, 32)", INK, 11, mono=True, weight="bold")
arrow(ax, (X, 0.55), (X, 0.97))
first = ["layer norm", "attention", "dropout"]
mid = step(1.0, first, ["04_01", "03_04", "03_03"], "x")
tag(ax, X + 0.5, mid + 0.4, "x′", INK, 11)
arrow(ax, (X, mid + 0.27), (X, mid + 0.7))
parts = ["layer norm", "feed-forward", "dropout"]
top = step(mid + 0.7, parts, ["04_01", "04_02", "03_03"], "x′")
arrow(ax, (X, top + 0.27), (X, top + 0.75), color=ACCENT, width=2.2)
text = "output   (1, 5, 32): the same shape as x"
tag(ax, X, top + 1.05, text, ACCENT, 11, weight="bold")
plt.show()

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ First, a warm-up: how do `view` and `transpose` cut the numbers into heads?

The class in step 1 does it in one line, on tensors with 4 dimensions. That's a big jump, so let's do it slowly on numbers we can read. Take `ἄνδρα` with only **4** numbers per letter, and number them 0 to 19 so we can follow each one. The shape is `(1, 5, 4)`: 1 text, 5 letters, 4 numbers per letter. We want 2 heads, so each head gets 2 of the 4 numbers:

```
            head 0    head 1
letter ἄ:  [  0   1 |   2   3 ]
letter ν:  [  4   5 |   6   7 ]
letter δ:  [  8   9 |  10  11 ]
letter ρ:  [ 12  13 |  14  15 ]
letter α:  [ 16  17 |  18  19 ]
```

Two moves get us there:

- `view(1, 5, 2, 2)` cuts each letter's 4 numbers into 2 pairs. The shape reads `(text, letters, heads, numbers per head)`. No number moves. Only the grouping changes.
- `transpose(1, 2)` swaps the letters and the heads, so the shape reads `(text, heads, letters, numbers per head)`. With the heads in front, each head holds its own small `(5, 2)` table, and the attention maths of chapter 3 runs on each one.

👉 **Predict first:** after the two moves, which numbers will head 0 hold for the letter `ν`?

In [ ]:
import torch

x = torch.arange(20).view(1, 5, 4)  # 1 text, 5 letters, 4 numbers per letter
print("start:          ", tuple(x.shape))

cut = x.view(1, 5, 2, 2)  # each letter's 4 numbers as 2 pairs
print("after view:     ", tuple(cut.shape), " (text, letters, heads, per head)")

heads = cut.transpose(1, 2)  # swap letters and heads
print("after transpose:", tuple(heads.shape), " (text, heads, letters, per head)")
print("head 0:", heads[0, 0].tolist())
print("head 1:", heads[0, 1].tolist())

back = heads.transpose(1, 2).reshape(1, 5, 4)  # swap back, glue the pairs together
print("joined again:   ", tuple(back.shape), " same as x?", torch.equal(back, x))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(1, 5, 4)` becomes `(1, 5, 2, 2)` and then `(1, 2, 5, 2)`. Head 0 holds the left half of every letter (`[4, 5]` for `ν`), and head 1 the right half (`[6, 7]`), just like the picture. Swapping back and gluing the pairs together (`reshape`) gives exactly the tensor we started with, so nothing was lost or mixed up.

The class below does the same with `(1, 5, 32)`: 2 heads of 16 numbers. Its `split` is the `view` plus `transpose`, and its last line is the swap back plus `reshape`.

</details>

### ❓ Your turn: can you cut 6 numbers into 3 heads?

Same trick, other sizes. Each of the 5 letters now has 6 numbers, and we want 3 heads with 2 numbers each. The result should have the shape `(1, 3, 5, 2)`: text, heads, letters, numbers per head.

👉 **Predict first:** head 0 should hold the first 2 numbers of every letter. Letter 0 has `0 1 2 3 4 5` and letter 1 has `6 7 8 9 10 11`. What are the first two rows of head 0?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
six = torch.arange(30).view(1, 5, 6)  # 1 text, 5 letters, 6 numbers per letter

# 👉 do: cut each letter's 6 numbers into 3 heads of 2, then put the heads
#        before the letters
three_heads = None

In [ ]:
from homer_ai.lessons import check

check(
    "the shape of the 3 heads",
    three_heads,
    test=lambda heads: tuple(heads.shape) == (1, 3, 5, 2),
    expected="(1, 3, 5, 2)",
    show=lambda heads: tuple(heads.shape),
    coach="(1, 5, 3, 2) means the cut is done but the swap is missing: add "
    ".transpose(1, 2).",
)
check(
    "head 0",
    three_heads,
    six.view(1, 5, 3, 2).transpose(1, 2),
    show=lambda heads: heads[0, 0].tolist(),
    coach="Right shape, wrong numbers? Then you wrote view(1, 3, 5, 2) in one go. "
    "view only re-cuts the numbers in the order they are in. It can't swap two "
    "dimensions. Cut with view(1, 5, 3, 2) first, then swap with transpose(1, 2).",
)

<details><summary>💡 Hint</summary>

The warm-up did `x.view(1, 5, 2, 2)` and then `.transpose(1, 2)`. Only the two sizes in the `view` change.

</details>

<details><summary>✅ Answer</summary>

```python
six = torch.arange(30).view(1, 5, 6)  # 1 text, 5 letters, 6 numbers per letter

three_heads = six.view(1, 5, 3, 2).transpose(1, 2)
```

Head 0 is `[[0, 1], [6, 7], [12, 13], [18, 19], [24, 25]]`: the first two numbers of each letter. Try `six.view(1, 3, 5, 2)` to see the trap. The shape is right, but head 0 comes out as `[[0, 1], [2, 3], [4, 5], …]`, which is all of letter 0 and the start of letter 1, cut up. No error, just wrong numbers.

</details>

### ❓ Step 1: attention for several heads, in one class

In [03_04](../03_attention/03_04_many_heads.ipynb) we built two heads by hand and joined them. Here every head is computed at once. The idea is the same:

```
(5 letters, 32 numbers) → cut the 32 into heads → (heads, 5, 16 numbers) → scores, mask, softmax, mix for each head → join → output projection
```

The cutting and joining are the moves of the warm-up: `view` re-cuts the same numbers into a new shape, `transpose` swaps two directions, and `reshape` re-cuts again after the swap back. The extra first dimension is the **batch** of [04_01](04_01_layer_norm.ipynb) (the texts processed together, here 1), so the tensors have 4 dimensions: `(batch, heads, letters, numbers per head)`.

The causal mask is saved with `register_buffer`, so it travels with the model but has no weights. Here it's a table of `True`/`False`, and `masked_fill` writes `-inf` into the `True` places. You get the same result as adding the `-inf` table of [03_03](../03_attention/03_03_hiding_the_future.ipynb). The difference is that a `True`/`False` table can be cut down to the length of the text (`mask[:length, :length]`), and it's the form the book and GPT code use. We also add the output projection, one extra 32 × 32 layer that the book puts after the join so the heads can mix their results.

👉 **Predict first:** with `n_heads=2`, how many numbers does one head use per letter?

In [ ]:
import torch
from torch import nn


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_length, drop_rate):
        super().__init__()
        self.n_heads = n_heads
        self.head_size = emb_dim // n_heads  # 32 // 2 = 16
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.out_proj = nn.Linear(emb_dim, emb_dim)  # the output projection
        self.dropout = nn.Dropout(drop_rate)
        mask = torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        self.register_buffer("mask", mask)  # True = hidden

    def forward(self, x):
        batch, length, emb_dim = x.shape
        # cut the 32 numbers into heads: into (batch, heads, length, 16)
        def split(t):
            return t.view(batch, length, self.n_heads, self.head_size).transpose(1, 2)

        q, k, v = split(self.query(x)), split(self.key(x)), split(self.value(x))
        scores = q @ k.transpose(2, 3) / self.head_size**0.5
        scores = scores.masked_fill(self.mask[:length, :length], float("-inf"))
        weights = self.dropout(torch.softmax(scores, dim=-1))
        mixed = weights @ v  # (batch, heads, length, 16)
        context = mixed.transpose(1, 2).reshape(batch, length, emb_dim)  # join
        return self.out_proj(context)


attention = MultiHeadAttention(32, n_heads=2, context_length=24, drop_rate=0.0)
print("head size:", attention.head_size)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

16 numbers per head, and 2 heads × 16 = 32, as in [03_04](../03_attention/03_04_many_heads.ipynb). Everything else in `forward` is the scores, mask, softmax and mix of chapter 3, with one spare direction for the heads.

</details>

### ❓ Is this really the same as the two heads of 03_04?

The batched code is shorter but harder to read, so let's prove it. Below is the 03_04 way: a loop with one small attention per head (rows 0-15 of each weight table make head 1's numbers, rows 16-31 make head 2's), then `torch.cat` to join. We give it the **same weights** as the batched class, and compare after the output projection.

👉 **Predict first:** will `torch.allclose` say the two agree?

In [ ]:
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
print("vocabulary size:", tokenizer.vocab_size)

In [ ]:
torch.manual_seed(0)
embedding = nn.Embedding(tokenizer.vocab_size, 32)  # a new, untrained table
ids = torch.tensor([tokenizer.encode("ἄνδρα")])  # shape (1, 5): 1 text, 5 letters
x = embedding(ids)  # shape (1, 5, 32): 5 letters, 32 numbers each
print("ids:", tuple(ids.shape), " x:", tuple(x.shape))

In [ ]:
def by_loop(x, attention):
    # x has shape (letters, 32): one text, so no batch dimension, as in chapter 3
    size = attention.head_size
    contexts = []
    for h in range(attention.n_heads):
        rows = slice(h * size, (h + 1) * size)  # this head's 16 of the 32 numbers
        Q = x @ attention.query.weight[rows].T
        K = x @ attention.key.weight[rows].T
        V = x @ attention.value.weight[rows].T
        scores = (Q @ K.T) / size**0.5
        hidden = torch.triu(torch.ones(len(x), len(x)) * float("-inf"), diagonal=1)
        scores = scores + hidden
        contexts.append(torch.softmax(scores, dim=-1) @ V)
    return attention.out_proj(torch.cat(contexts, dim=-1))  # join, then project


attention.eval()  # no dropout
with torch.no_grad():
    batched = attention(x)[0]  # [0]: drop the batch dimension
    looped = by_loop(x[0], attention)
print("batched equals the loop of 03_04?", torch.allclose(batched, looped, atol=1e-6))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True`. The 4-dimensional code and the loop give the same numbers. The batched version just does both heads at once.

</details>

### ❓ Step 2: the feed-forward step, and the block itself

The feed-forward step is the same as in [04_02](04_02_feed_forward.ipynb). The block then follows the picture above, line by line: `x = x + dropout(attention(norm1(x)))`, then `x = x + dropout(feed_forward(norm2(x)))`.

👉 **Predict first:** the block takes `(1, 5, 32)` for the five letters of `ἄνδρα`. What shape comes out?

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim), nn.GELU(), nn.Linear(4 * emb_dim, emb_dim)
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg["emb_dim"], cfg["n_heads"], cfg["context_length"], cfg["drop_rate"]
        )
        self.feed_forward = FeedForward(cfg["emb_dim"])
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm1(x)))  # step 1 + shortcut
        x = x + self.dropout(self.feed_forward(self.norm2(x)))  # step 2 + shortcut
        return x


GPT_CONFIG = {
    "vocab_size": 137, "context_length": 24, "emb_dim": 32,
    "n_heads": 2, "n_layers": 2, "drop_rate": 0.1,
}
block = TransformerBlock(GPT_CONFIG)
print(block)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The block lists its parts: the attention (with its three layers, the output projection and dropout), the feed-forward, two layer norms and a dropout. A `cfg` dictionary is all it needs. Every size comes from [the config](04_00_chapter_overview.ipynb) and nowhere else.

</details>

### ❓ Step 3: does the shape stay the same?

Take the five letters of `ἄνδρα` as untrained embedding rows (the `x` from above), and send them through the block.

In [ ]:
block.eval()  # switch dropout off, so the answers are the same every time
out = block(x)
print("in :", tuple(x.shape))
print("out:", tuple(out.shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(1, 5, 32)` in and `(1, 5, 32)` out. Same shape, so the output of one block can be the input of the next, again and again.

</details>

### ❓ Step 4: does the block still hide the future?

A block has to keep the rule from [03_03](../03_attention/03_03_hiding_the_future.ipynb). So let's test it: change only the **last** letter, and see whether the output for the first four letters changes.

👉 **Predict first:** will the first four rows of the output change?

In [ ]:
changed = x.clone()
changed[0, 4] = torch.randn(32)  # a different last letter

same_before_last = torch.allclose(block(x)[0, :4], block(changed)[0, :4], atol=1e-6)
same_last = torch.allclose(block(x)[0, 4], block(changed)[0, 4], atol=1e-6)
print("first 4 letters unchanged?", same_before_last)
print("last letter unchanged?    ", same_last)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first four are unchanged (nothing leaks backwards) and the last one changed, as it should. The feed-forward and the norms work letter by letter, so only attention could leak, and the mask stops it.

</details>

### ❓ Step 5: how many weights?

Let's count the real ones, part by part. The attention has the three 32 × 32 layers (3,072, as in our model) plus the 32 × 32 output projection with its 32 biases.

👉 **Predict first:** is the whole block bigger than our entire `model.py` (11,840)?

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())


for name in ["attention", "feed_forward", "norm1", "norm2"]:
    print(f"{name:13} {count(getattr(block, name)):>6,}")
print(f"{'one block':13} {count(block):>6,}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Attention 4,128, feed-forward 8,352, two norms at 64 each. That's 12,608 for one block, a bit more than our whole `model.py`. Feed-forward is the biggest part (about two thirds). And the mask is a buffer with no weights, so it isn't counted.

</details>

## 5. Recap

- A block is `x = x + attention(norm(x))`, then `x = x + feed_forward(norm(x))`. The shape in equals the shape out.
- Our block with 32 numbers and 2 heads has 12,608 weights (attention 4,128, feed-forward 8,352, two norms 64 each) and doesn't leak the future.
- Because the shape is unchanged, blocks can be stacked. That's the next notebook.

**You can now:**

- cut a row into heads with `view`, and swap dimensions with `transpose`
- write a block's `forward`: two shortcuts, each around norm and then a layer
- check that a block keeps the shape and hides the future

**Next:** [04_05: The full model](04_05_the_full_model.ipynb): stacking blocks and writing letters.